In [1]:
!pip install --upgrade --quiet google-cloud-aiplatform[agent_engines,adk]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 75.6 MB/s eta 0:00:00


## Challenge-5-Deploy Agent

In [2]:
import os

PROJECT_ID = "qwiklabs-gcp-04-91465eedb012"
REGION = "us-central1"
MODEL="gemini-3.5-flash"

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = REGION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"

STAGING_BUCKET = f"gs://{PROJECT_ID}-agent-staging"

In [19]:


from google.adk import Agent
import vertexai
from google.adk.tools import google_search

vertexai.init(
    project=PROJECT_ID,
    location=REGION,
    staging_bucket=STAGING_BUCKET,
)

search_agent = Agent(
    name="Google",
    description=("You can search Google…"),
    instruction=(
    """You are a helpful Chatbot."""
    ),
    tools=[google_search]
)


In [20]:
from vertexai.preview.reasoning_engines import AdkApp
app = AdkApp(agent=search_agent)

In [21]:
try:
    print("test local stream query...")
    responses = app.stream_query(
        user_id="local-user",
        message="Give me the news highlights in the world of sports.",
    )
    for response in responses:
        print(response)

except Exception as e:
    print(f"Local test failed with error: {e}")

Starting local stream query...
{'model_version': 'gemini-2.5-flash', 'content': {'parts': [{'text': 'Here\'s a roundup of recent sports highlights:\n\n**Football (Soccer)**\n\nThe 2026 FIFA World Cup is currently underway, with significant attention on the ongoing matches. Kylian Mbappé has been a central figure, rewriting FIFA World Cup record books with a never-before-seen feat, although he recently suffered a "slight" ankle injury after scoring his 8th World Cup goal. France defeated Morocco 2-0 in the quarterfinals, with Mbappé contributing a goal and an assist, and Ousmane Dembélé scoring from outside the box to extend France\'s lead. Argentina also saw a comeback against Egypt, with Lionel Messi equalizing and Enzo Fernández scoring the go-ahead goal. Rafael Márquez has been named the Mexican National Team Head Coach for the 2030 World Cup, and Justin Bieber is set to co-headline the 2026 World Cup Final Halftime Show. Greater Miami and Miami Beach are hosting World Cup matches a

In [27]:
from vertexai import agent_engines
remote_agent = agent_engines.create(
            app,
            display_name="google_searcher",
            requirements=["google-cloud-aiplatform[agent_engines,adk]","google-adk"]
        )

INFO:vertexai.agent_engines:Identified the following requirements: {'pydantic': '2.12.3', 'cloudpickle': '3.1.2', 'google-cloud-aiplatform': '1.160.0'}
INFO:vertexai.agent_engines:The following requirements are appended: {'cloudpickle==3.1.2', 'pydantic==2.12.3'}
INFO:vertexai.agent_engines:The final list of requirements: ['google-cloud-aiplatform[agent_engines,adk]', 'google-adk', 'cloudpickle==3.1.2', 'pydantic==2.12.3']
INFO:vertexai.agent_engines:Using bucket qwiklabs-gcp-04-91465eedb012-agent-staging
INFO:vertexai.agent_engines:Wrote to gs://qwiklabs-gcp-04-91465eedb012-agent-staging/agent_engine/agent_engine.pkl
INFO:vertexai.agent_engines:Writing to gs://qwiklabs-gcp-04-91465eedb012-agent-staging/agent_engine/requirements.txt
INFO:vertexai.agent_engines:Creating in-memory tarfile of extra_packages
INFO:vertexai.agent_engines:Writing to gs://qwiklabs-gcp-04-91465eedb012-agent-staging/agent_engine/dependencies.tar.gz
INFO:vertexai.agent_engines:Creating AgentEngine
INFO:vertexai.a

In [36]:
print(f'remote_agent:{remote_agent}')

for event in remote_agent.stream_query(
  user_id="agent-engine-test-user",
  message="Give me the news highlights in the world of sports.",
):
  print(event)

remote_agent:<vertexai.agent_engines._agent_engines.AgentEngine object at 0x7c8a6c919280> 
resource name: projects/112530703323/locations/us-central1/reasoningEngines/7112976565327101952
